In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from statsmodels.stats.contingency_tables import mcnemar

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")

## 13. Model Evaluation & Comparison

Comprehensive evaluation: ROC, PR curves, threshold analysis, and statistical significance.

In [ ]:
path = Path("../data/processed/train_features.parquet")
if not path.exists():
    path = Path("../data/interim/train_merged.parquet")
train = pd.read_parquet(path)

exclude_cols = ["TransactionID", "isFraud", "TransactionDT"]
feature_cols = [c for c in train.select_dtypes(include="number").columns if c not in exclude_cols]
X = train[feature_cols]
y = train["isFraud"]

X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"Train: {X_tr.shape}, Val: {X_va.shape}")

### 13.1 Load Models & Predict

In [ ]:
# Train models
dtrain_lgb = lgb.Dataset(X_tr, label=y_tr)
dval_lgb = lgb.Dataset(X_va, label=y_va, reference=dtrain_lgb)
model_lgb = lgb.train(
    {"objective": "binary", "metric": "auc", "num_leaves": 31, "learning_rate": 0.1, "verbose": -1, "random_state": 42},
    dtrain_lgb, num_boost_round=100, valid_sets=[dval_lgb], callbacks=[lgb.early_stopping(15, verbose=False)]
)

dtrain_xgb = xgb.DMatrix(X_tr, label=y_tr)
dval_xgb = xgb.DMatrix(X_va, label=y_va)
model_xgb = xgb.train(
    {"objective": "binary:logistic", "eval_metric": "auc", "max_depth": 6, "learning_rate": 0.1, "verbosity": 0, "random_state": 42},
    dtrain_xgb, num_boost_round=100, evals=[(dval_xgb, "val")],
    callbacks=[xgb.callback.EarlyStopping(rounds=15, save_best=True, metric_name="auc", data_name="val")],
    verbose_eval=False
)

model_cb = CatBoostClassifier(iterations=100, learning_rate=0.1, depth=6, eval_metric="AUC", random_seed=42, verbose=False)
model_cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=15)

preds_lgb = model_lgb.predict(X_va, num_iteration=model_lgb.best_iteration)
preds_xgb = model_xgb.predict(dval_xgb)
preds_cb = model_cb.predict_proba(X_va)[:, 1]

### 13.2 ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, preds in [("LightGBM", preds_lgb), ("XGBoost", preds_xgb), ("CatBoost", preds_cb)]:
    fpr, tpr, _ = roc_curve(y_va, preds)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves comparison")
ax.legend(loc="lower right")
Path("../data/metadata").mkdir(parents=True, exist_ok=True)
plt.savefig("../data/metadata/roc_curves_comparison.png")
plt.show()

### 13.3 Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, preds in [("LightGBM", preds_lgb), ("XGBoost", preds_xgb), ("CatBoost", preds_cb)]:
    precision, recall, _ = precision_recall_curve(y_va, preds)
    pr_auc = average_precision_score(y_va, preds)
    ax.plot(recall, precision, label=f"{name} (PR-AUC = {pr_auc:.4f})")

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves comparison")
ax.legend(loc="lower left")
plt.show()

### 13.4 Confusion Matrix

In [ ]:
# LightGBM confusion matrix
cm_lgb = confusion_matrix(y_va, (preds_lgb >= 0.5).astype(int))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_lgb, display_labels=["Normal", "Fraud"])
disp.plot(cmap="Blues")
plt.title("LightGBM Confusion Matrix (threshold=0.5)")
plt.show()

### 13.5 McNemar's Test

In [ ]:
# Statistical significance test between LightGBM and XGBoost predictions
lgb_correct = ((preds_lgb >= 0.5).astype(int) == y_va)
xgb_correct = ((preds_xgb >= 0.5).astype(int) == y_va)

# Contingency table
n_00 = np.sum(~lgb_correct & ~xgb_correct)
n_01 = np.sum(~lgb_correct & xgb_correct)
n_10 = np.sum(lgb_correct & ~xgb_correct)
n_11 = np.sum(lgb_correct & xgb_correct)

table = [[n_11, n_10], [n_01, n_00]]
result = mcnemar(table, exact=False, correction=True)
print(f"McNemar statistic: {result.statistic:.3f}, p-value: {result.pvalue:.4e}")
if result.pvalue < 0.05:
    print("Difference in model accuracy is statistically significant.")
else:
    print("Difference in model accuracy is not statistically significant.")